In [ ]:
from datetime import datetime, timedelta

import numpy as np
from stonesoup.models.transition.linear import (
    CombinedLinearGaussianTransitionModel,
    ConstantVelocity,
)
from stonesoup.types.groundtruth import GroundTruthState

from nereus.stonesoup.movable import TowedArrayPlatform

np.random.seed(42)

SIM_PARAMS = {
    "start_time": datetime.now().replace(hour=0, minute=0, second=0, microsecond=0),
    "time_interval": timedelta(seconds=10),
    "num_steps": 180,
}

SHIP_PARAMS = {
    "start_vector": np.array([0, 1.82, 0, 1.82, -10.0, 0]), # 5 knots = 2.57 m/s
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.0001), ConstantVelocity(0.0001), ConstantVelocity(0)]
    ),
}

ARRAY_PARAMS = {
    "num_sensors": 1000,
    "tow_cable_length": 1800.0,
    "sensor_spacing": 0.75,
    "array_depth": -1000.0,
}


# Generate platform path

In [ ]:
init_state = GroundTruthState(
    SHIP_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"]
)

platform = TowedArrayPlatform(
    state=init_state,
    position_mapping=SHIP_PARAMS["position_mapping"],
    velocity_mapping=SHIP_PARAMS["velocity_mapping"],
    transition_model=SHIP_PARAMS["transition_model"],
    num_sensors=ARRAY_PARAMS["num_sensors"],
    cable_length=ARRAY_PARAMS["tow_cable_length"],
    sensor_spacing=ARRAY_PARAMS["sensor_spacing"],
    array_depth=ARRAY_PARAMS["array_depth"],
)

In [ ]:
for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    platform.move(new_time)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import HTML, display
from matplotlib.animation import FuncAnimation

plt.rcParams['animation.embed_limit'] = 100  # Set to 100 MB (default is 20 MB)

sns.set_theme(style="whitegrid")
fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection='3d')
# --------------------------------------------------

ship = platform.ship
sensors = platform.sensors

# For the ship, we get x, y, and z from indices [0, 2, 4]
ship_positions = np.array([s.state_vector[[0, 2, 4]].flatten() for s in ship.states])
# The follower paths are already 3D, so this is unchanged
follower_paths = [
    np.array([s.state_vector.flatten() for s in follower.states])
    for follower in sensors
]
# -----------------------------------------------

# Set plot limits dynamically for all 3 axes
all_x = np.concatenate([ship_positions[:, 0]] + [path[:, 0] for path in follower_paths])
all_y = np.concatenate([ship_positions[:, 1]] + [path[:, 1] for path in follower_paths])
all_z = np.concatenate([ship_positions[:, 2]] + [path[:, 2] for path in follower_paths])
ax.set_xlim(all_x.min() - 100, all_x.max() + 100)
ax.set_ylim(all_y.min() - 100, all_y.max() + 100)
ax.set_zlim(all_z.min() - 20, all_z.max() + 20)

ax.set_title("3D Towed Array Simulation")
ax.set_xlabel("X Position (m)")
ax.set_ylabel("Y Position (m)")
ax.set_zlabel("Depth (m)")

# Initialise artists for the animation in 3D
(point_ship,) = ax.plot([], [], [], "o", markersize=10, color="C0", label="Ship")
(line_ship,) = ax.plot([], [], [], "--", alpha=0.6, color="C0", label="Ship Path")
(line_array,) = ax.plot([], [], [], "o-", markersize=5, color="k", label="Towed Array")
ax.legend()

def update(frame):
    """Update the animation for each frame."""
    # Get the 3D position of all nodes at the current frame
    all_nodes_x = [ship_positions[frame, 0]] + [path[frame, 0] for path in follower_paths]
    all_nodes_y = [ship_positions[frame, 1]] + [path[frame, 1] for path in follower_paths]
    all_nodes_z = [ship_positions[frame, 2]] + [path[frame, 2] for path in follower_paths]

    point_ship.set_data_3d([all_nodes_x[0]], [all_nodes_y[0]], [all_nodes_z[0]])

    line_ship.set_data(ship_positions[: frame + 1, 0], ship_positions[: frame + 1, 1])
    line_ship.set_3d_properties(ship_positions[: frame + 1, 2])
    line_array.set_data(all_nodes_x, all_nodes_y)
    line_array.set_3d_properties(all_nodes_z)

    return point_ship, line_ship, line_array


ani = FuncAnimation(
    fig, update, frames=SIM_PARAMS["num_steps"], blit=False, interval=100
)

plt.close()

display(HTML(ani.to_jshtml()))

# Generate source path

In [ ]:
from stonesoup.movable import MovingMovable

from nereus.stonesoup.targets import AcousticTarget

TARGET_PARAMS = {
    "start_vector": np.array([10000, -7.28, 10000, 7.28, -10.0, 0]), # 20 knots = 10.29 m/s
    "position_mapping": [0, 2, 4],
    "velocity_mapping": [1, 3],
    "transition_model": CombinedLinearGaussianTransitionModel(
        [ConstantVelocity(0.1), ConstantVelocity(0.1), ConstantVelocity(0)]
    ),
    "amplitudes_upa": 10 ** (np.array([180.0, 165.0, 170.0]) / 20),
    "frequencies_hz": [125.0, 190.0, 285.0],
    "phases_rad": np.deg2rad([45.0, 125.0, 5.0])
}

# First, create the inner movable object
movable_component = MovingMovable(
    states=[GroundTruthState(
        TARGET_PARAMS["start_vector"], timestamp=SIM_PARAMS["start_time"]
    )],
    position_mapping=TARGET_PARAMS["position_mapping"],
    transition_model=TARGET_PARAMS["transition_model"]
)

# Now, create the main AcousticTarget, composing it from its parts
target = AcousticTarget(
    movable=movable_component,
    amplitudes_upa=TARGET_PARAMS["amplitudes_upa"],
    frequencies_hz=TARGET_PARAMS["frequencies_hz"],
    phases_rad=TARGET_PARAMS["phases_rad"]
)

for i in range(1, SIM_PARAMS["num_steps"]):
    new_time = SIM_PARAMS["start_time"] + i * SIM_PARAMS["time_interval"]
    target.move(new_time)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(projection='3d')

# --- Get the objects from your simulation ---
ship = platform.ship
sensors = platform.sensors

# --- Extract full 3D path data for all objects ---
ship_positions = np.array([s.state_vector[[0, 2, 4]].flatten() for s in ship.states])
follower_paths = [
    np.array([s.state_vector.flatten() for s in follower.states])
    for follower in sensors
]
target_positions = np.array(
    [s.state_vector[[0, 2, 4]].flatten() for s in target.states]
)

# --- Set plot limits dynamically to include all objects ---
all_x = np.concatenate([
    ship_positions[:, 0]] + \
        [path[:, 0] for path in follower_paths] + \
            [target_positions[:, 0]
])
all_y = np.concatenate([
    ship_positions[:, 1]] + \
        [path[:, 1] for path in follower_paths] + \
            [target_positions[:, 1]])
all_z = np.concatenate([
    ship_positions[:, 2]] + \
        [path[:, 2] for path in follower_paths] + \
            [target_positions[:, 2]])
ax.set_xlim(all_x.min() - 100, all_x.max() + 100)
ax.set_ylim(all_y.min() - 100, all_y.max() + 100)
ax.set_zlim(all_z.min() - 20, all_z.max() + 20)

ax.set_title("3D Towed Array and Target Simulation")
ax.set_xlabel("X Position (m)")
ax.set_ylabel("Y Position (m)")
ax.set_zlabel("Depth (m)")

# --- Initialise artists for the animation ---
(point_ship,) = ax.plot([], [], [], "o", markersize=8, color="royalblue", label="Ship")
(line_ship,) = ax.plot([], [], [], "--", alpha=0.6, color="C0", label="Ship Path")
(line_array,) = ax.plot(
    [], [], [], "o-", markersize=3, color="black", label="Towed Array"
)
(point_target,) = ax.plot([], [], [], "o", markersize=8, color="red", label="Target")
(line_target,) = ax.plot([], [], [], "--", alpha=0.6, color="red", label="Target Path")
ax.legend()

def update(frame):
    """Update the animation for each frame."""
    # Update ship and array positions
    all_nodes_x = [ship_positions[frame, 0]] + \
        [path[frame, 0] for path in follower_paths]
    all_nodes_y = [ship_positions[frame, 1]] + \
        [path[frame, 1] for path in follower_paths]
    all_nodes_z = [ship_positions[frame, 2]] + \
        [path[frame, 2] for path in follower_paths]
    point_ship.set_data_3d([all_nodes_x[0]], [all_nodes_y[0]], [all_nodes_z[0]])
    line_array.set_data_3d(all_nodes_x, all_nodes_y, all_nodes_z)
    line_ship.set_data(ship_positions[:frame + 1, 0], ship_positions[:frame + 1, 1])
    line_ship.set_3d_properties(ship_positions[:frame + 1, 2])

    # Update the target's current position
    point_target.set_data_3d(
        [target_positions[frame, 0]],
        [target_positions[frame, 1]],
        [target_positions[frame, 2]]
    )
    # Update the target's path history
    line_target.set_data(
        target_positions[:frame + 1, 0], target_positions[:frame + 1, 1]
    )
    line_target.set_3d_properties(target_positions[:frame + 1, 2])

    return point_ship, line_ship, line_array, point_target, line_target

# Create the animation
ani = FuncAnimation(
    fig, update, frames=SIM_PARAMS["num_steps"], blit=False, interval=100
)
plt.close()

# Display in the notebook
display(HTML(ani.to_jshtml()))

In [ ]:
from scipy.signal.windows import get_window

from nereus.stonesoup.beamformers import DelayAndSumBeamformer
from nereus.stonesoup.propagators import CylindricalAcousticPropagationModel
from nereus.stonesoup.signals import AcousticSignalModel, ColouredNoise
from nereus.stonesoup.sound_speed_profiles import Munk

PROP_PARAMS = {
    "ssp": Munk()
}

SIGNAL_PARAMS = {
    "duration_s": 1.0,
    "sampling_rate_hz": 1000,
    "noise_spectral_exponent": -1,  # -1 for pink noise, 0 for white noise, etc.
    "noise_amplitude_upa": 10 ** (80 / 20)
}

BF_PARAMS = {
    "shading_window": get_window("blackman", ARRAY_PARAMS["num_sensors"]),
    "domain": "frequency",  # 'frequency' or 'time'
}

propagator = CylindricalAcousticPropagationModel(
    ssp=PROP_PARAMS["ssp"],
)

signal_model = AcousticSignalModel(
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
)

noise_model = ColouredNoise(
    spectral_exponent=SIGNAL_PARAMS["noise_spectral_exponent"],
    amplitude_upa=SIGNAL_PARAMS["noise_amplitude_upa"],
    duration_s=SIGNAL_PARAMS["duration_s"],
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
)

beamformer = DelayAndSumBeamformer(
    sampling_rate_hz=SIGNAL_PARAMS["sampling_rate_hz"],
    shading_window=BF_PARAMS["shading_window"],
    domain=BF_PARAMS["domain"],
)